# Una IA pide una herramienta

En este cuaderno haces **a mano** lo que hace un agente de IA cuando usa un servidor MCP: conectarse, leer qué herramientas hay, llamarlas y leer lo que devuelven. Al final ves el JSON con el que un modelo de verdad pidió una herramienta.

El servidor es **Madrid ahora**: dos herramientas con datos en directo (el tiempo de este momento y el precio de la luz de hoy).

**Qué necesitas:** la URL y el token que están en el proyector. Nada más: no se instala nada en tu ordenador.

**Cómo se usa:** cada celda gris tiene un botón de play a la izquierda (o pulsa Mayúsculas + Intro). Ejecútalas en orden, de arriba abajo.

## 1. Prepara el cuaderno

Instala la librería oficial de MCP para Python, la versión 2.3.0. Tarda unos 30 segundos.

Si salen avisos en rojo sobre "dependency conflicts", no pasa nada: sigue. Si Colab te pide reiniciar la sesión ("Restart session"), acepta y sigue desde la celda siguiente.

In [ ]:
%pip install -q mcp==2.3.0

Esta celda prepara dos ayudantes para no repetir código en cada paso:

- `conectar()` abre la conexión con el servidor, con tu token en la cabecera `Authorization`.
- `llamar(herramienta, argumentos)` llama a una herramienta y te devuelve lo que contesta.

No hace falta que la entiendas entera: ejecútala y sigue.

In [ ]:
import inspect
import json
from getpass import getpass

import httpx2
from mcp import Client
from mcp.client.streamable_http import streamable_http_client


def conectar(token=None):
    """Abre la conexión con el servidor MCP, con el token en la cabecera."""
    cabeceras = {"Authorization": f"Bearer {token or TOKEN}"}
    http = httpx2.AsyncClient(headers=cabeceras, timeout=120)  # 120 s: Render gratis tarda en despertar
    return Client(streamable_http_client(URL, http_client=http))


async def llamar(herramienta, argumentos=None):
    """Llama a una herramienta. Devuelve lo que contesta, o enseña el error tal como lo lee la IA."""
    async with conectar() as cliente:
        resultado = await cliente.call_tool(herramienta, argumentos or {})
    texto = resultado.content[0].text
    if resultado.is_error:
        print("ERROR. Esto es lo que leería la IA:\n")
        print(texto)
        return None
    return json.loads(texto)


print("Listo")

## 2. Conéctate

Escribe la URL y el token del proyector. El token no se ve al escribirlo y **no se guarda en el cuaderno**: si compartes el cuaderno, el token no va dentro. No lo imprimas en ninguna celda.

In [ ]:
URL = input("URL del MCP (acaba en /mcp): ").strip()
TOKEN = getpass("Token de la clase: ").strip()

Comprueba que conecta. La primera vez puede tardar hasta un minuto: si el servidor estaba dormido, Render lo despierta.

In [ ]:
async with conectar() as cliente:
    print("Conectado a:", cliente.server_info.name)
    print()
    print("Instrucciones del servidor para la IA:")
    print(cliente.instructions)

Si ha salido un error largo en rojo, mira la URL (tiene que acabar en `/mcp`) y vuelve a ejecutar la celda de arriba. Para ver qué dice el servidor, ejecuta esta:

In [ ]:
respuesta = httpx2.post(URL, headers={"Authorization": f"Bearer {TOKEN}"}, timeout=120)
print(respuesta.status_code, respuesta.text[:300])

Si sale `401`, el token está mal copiado: vuelve a la celda de la URL y el token. Si sale otro número (`400`, `406`...), el token es bueno: el servidor solo se queja de que no le has mandado un mensaje MCP de verdad.

## 3. Lee lo que ofrece

Un modelo de IA **no lee el código del servidor**. Lee esto: el nombre de cada herramienta, su descripción y sus parámetros. Con eso decide si la usa y cómo.

In [ ]:
async with conectar() as cliente:
    herramientas = (await cliente.list_tools()).tools

for h in herramientas:
    print("HERRAMIENTA:", h.name)
    print(inspect.cleandoc(h.description))
    print("PARÁMETROS:", json.dumps(h.input_schema.get("properties", {}), ensure_ascii=False, indent=2))
    print("-" * 60)

**Antes de seguir, piensa:** si fueras la IA y te preguntaran "¿llevo paraguas si salgo ahora por Getafe?", ¿qué herramienta elegirías y con qué parámetro? ¿Qué frase de la descripción te lo ha dicho?

## 4. Llama a una herramienta a mano

Esto es lo que hace el agente cuando el modelo le pide una herramienta. Lo que vuelve es un texto en JSON; aquí lo ves ordenado como un diccionario de Python.

In [ ]:
tiempo = await llamar("tiempo_ahora", {"lugar": "Madrid"})
tiempo

Cambia `"Madrid"` por tu municipio o tu barrio y vuelve a ejecutar la celda de arriba.

Ahora la otra herramienta. No tiene parámetros: siempre es el precio de hoy.

In [ ]:
luz = await llamar("precio_luz_hoy")

print(f"Ahora ({luz['hora_actual']}): {luz['precio_ahora_eur_kwh']} €/kWh")
print("Las tres horas más baratas de hoy:")
for hora in luz["tres_horas_mas_baratas"]:
    print(f"   {hora['tramo']}: {hora['eur_kwh']} €/kWh")
print(f"La más cara: {luz['hora_mas_cara']['tramo']}: {luz['hora_mas_cara']['eur_kwh']} €/kWh")

**Piensa:** un chat sin herramientas no puede saber estas cifras: son de hoy. Si se las preguntas, o te dice que no lo sabe o se las inventa.

## 5. Encadena dos herramientas

Aquí usas el resultado de una herramienta para decidir si llamas a la otra. Lo decides tú, con un `if`.

In [ ]:
tiempo = await llamar("tiempo_ahora", {"lugar": "Getafe"})
print(f"En {tiempo['lugar']} hay {tiempo['temperatura_c']} °C (sensación de {tiempo['sensacion_termica_c']} °C).")

if tiempo["temperatura_c"] < 12:
    luz = await llamar("precio_luz_hoy")
    barata = luz["tres_horas_mas_baratas"][0]
    print(f"Hace frío: la hora más barata de hoy para la calefacción es {barata['tramo']}, a {barata['eur_kwh']} €/kWh.")
else:
    print("No hace frío: no hace falta mirar la luz.")

Un agente hace lo mismo, pero **la decisión la toma el modelo**: lee el primer resultado y escribe la siguiente petición. Nadie programa el `if`.

## 6. Provoca un error y léelo

Un lugar que no existe. El servidor contesta con un mensaje pensado para la IA: dice qué ha pasado **y qué hacer**.

In [ ]:
await llamar("tiempo_ahora", {"lugar": "Narnia"})

Ahora un lugar de una sola letra. Este error no lo escribe el servidor: lo escribe la librería que comprueba los parámetros (`min_length=2`), y sale en inglés.

In [ ]:
await llamar("tiempo_ahora", {"lugar": "X"})

Y un error en la puerta: un token que no es el bueno. Aquí no llega ni a la herramienta: el servidor contesta `401`.

In [ ]:
try:
    async with conectar(token="esto-no-es-el-token") as cliente:
        await cliente.list_tools()
except Exception as error:
    print("La librería no conecta, pero no dice por qué:", type(error).__name__)

respuesta = httpx2.post(URL, headers={"Authorization": "Bearer esto-no-es-el-token"}, timeout=120)
print("Lo que dice el servidor:", respuesta.status_code, respuesta.text)

**Piensa:** ¿cuál de los tres mensajes ayuda más a una IA a corregirse sola en el segundo intento? ¿Qué le falta al de la librería?

## 7. Así pide un modelo una herramienta (grabado)

Esto pasó de verdad el 25/11/2026 a las 23:02. Le preguntamos a **Claude Code** (modelo `claude-sonnet-5-5`), conectado a Madrid ahora:

> ¿Hace frío ahora en Getafe? ¿Y a qué hora de hoy es más barata la luz? Responde en tres líneas.

Lo grabamos en cuatro momentos. Ejecuta la celda para verlo (la salida ya está guardada, por si no la ejecutas).

In [ ]:
GRABADO = {
    "1_el_modelo_escribe": [
        {"type": "tool_use", "id": "toolu_01Qupd91UPF6K12chH5CVLnU",
         "name": "mcp__madrid_ahora__tiempo_ahora", "input": {"lugar": "Getafe"}},
        {"type": "tool_use", "id": "toolu_01JhCMLGU5s6mXboScCKxg6B",
         "name": "mcp__madrid_ahora__precio_luz_hoy", "input": {}},
    ],
    "2_el_agente_manda_al_servidor": {
        "jsonrpc": "2.0", "id": 3, "method": "tools/call",
        "params": {"name": "tiempo_ahora", "arguments": {"lugar": "Getafe"}},
    },
    "3_el_servidor_devuelve": {
        "lugar": "Getafe", "provincia": "Provincia de Madrid", "medido_a_las": "23:00",
        "temperatura_c": 6.7, "sensacion_termica_c": 3.5, "cielo": "despejado",
        "lluvia_mm": 0.0, "viento_kmh": 10.9, "humedad_pct": 80, "es_de_dia": False,
        "fuente": "Open-Meteo",
    },
    "4_el_modelo_responde": (
        "Sí, hace frío: son las 23:00 y hay 6,7 °C en Getafe, con sensación de 3,5 °C. "
        "El cielo está despejado, hay algo de viento (11 km/h) y la humedad es del 80 %.\n"
        "La hora más barata de hoy fue de 14:00 a 15:00, a 0,0553 €/kWh. Ya ha pasado, "
        "porque ahora son las 23:00.\n"
        "Ahora mismo la luz está a 0,2766 €/kWh, por encima de la media del día (0,1957). "
        "Lo más barato que queda es de madrugada, de 04:00 a 06:00, a unos 0,10 €/kWh, "
        "pero esa franja ya cuenta como el día siguiente."
    ),
}

for paso, contenido in GRABADO.items():
    print("=" * 12, paso.replace("_", " "))
    print(contenido if isinstance(contenido, str) else json.dumps(contenido, ensure_ascii=False, indent=2))

============ 1 el modelo escribe
[
  {
    "type": "tool_use",
    "id": "toolu_01Qupd91UPF6K12chH5CVLnU",
    "name": "mcp__madrid_ahora__tiempo_ahora",
    "input": {
      "lugar": "Getafe"
    }
  },
  {
    "type": "tool_use",
    "id": "toolu_01JhCMLGU5s6mXboScCKxg6B",
    "name": "mcp__madrid_ahora__precio_luz_hoy",
    "input": {}
  }
]
============ 2 el agente manda al servidor
{
  "jsonrpc": "2.0",
  "id": 3,
  "method": "tools/call",
  "params": {
    "name": "tiempo_ahora",
    "arguments": {
      "lugar": "Getafe"
    }
  }
}
============ 3 el servidor devuelve
{
  "lugar": "Getafe",
  "provincia": "Provincia de Madrid",
  "medido_a_las": "23:00",
  "temperatura_c": 6.7,
  "sensacion_termica_c": 3.5,
  "cielo": "despejado",
  "lluvia_mm": 0.0,
  "viento_kmh": 10.9,
  "humedad_pct": 80,
  "es_de_dia": false,
  "fuente": "Open-Meteo"
}
============ 4 el modelo responde
Sí, hace frío: son las 23:00 y hay 6,7 °C en Getafe, con sensación de 3,5 °C. El cielo está despejado, hay

Fíjate en tres cosas:

1. **El modelo no ejecuta nada.** Escribe dos peticiones (`tool_use`) con el nombre de la herramienta y sus argumentos. `"lugar": "Getafe"` lo ha sacado de tu pregunta, guiado por la descripción del parámetro.
2. **El agente traduce cada petición a MCP** (`tools/call`) y la manda al servidor. El nombre largo `mcp__madrid_ahora__tiempo_ahora` es el que Claude Code le pone por dentro: servidor más herramienta.
3. **La respuesta usa las cifras que devolvió el servidor**, y razona con ellas: la hora más barata ya ha pasado. También se equivoca: dice que lo más barato que queda es de 04:00 a 06:00, pero esos precios son de hoy y esas horas ya han pasado. Las cifras vienen de la herramienta; el razonamiento, del modelo.

Lo hemos recortado un poco: Claude Code añade metadatos a cada `tools/call` (su versión y sus capacidades) y, antes de pedir las herramientas, hizo una petición interna para cargarlas.

Ahora **haz tú de agente**: manda al servidor la misma petición que escribió el modelo.

In [ ]:
peticion = GRABADO["2_el_agente_manda_al_servidor"]["params"]
await llamar(peticion["name"], peticion["arguments"])

Las cifras no coinciden con las grabadas: son las de ahora.

**Lo siguiente es tu agente.** Los últimos 10 minutos de la práctica son el paso 0 de la guía `tu-mcp.md`: conectas tu agente a esta misma URL con este mismo token y le haces una pregunta.

## 8. Opcional: en vivo con Gemini

Si tienes una clave de Google AI Studio (https://aistudio.google.com), puedes ver a un modelo **decidir en directo** qué herramienta pedir. La celda está comentada (cada línea empieza por `#`) para que no falle a quien no tiene clave.

Para usarla: selecciona todas sus líneas y pulsa Ctrl + / (en Mac, Cmd + /) para quitar los `#`. La clave se escribe en un recuadro y no se guarda en el cuaderno.

Esta celda **no está probada**: no teníamos clave al prepararla. Si falla el nombre del modelo, cámbialo por uno de la lista de AI Studio.

In [ ]:
# %pip install -q google-genai
# from google import genai
# from google.genai import types
#
# clave = getpass("Clave de AI Studio: ").strip()
# gemini = genai.Client(api_key=clave)
#
# # Las herramientas de Madrid ahora, tal como las ve el modelo: nombre, descripción y parámetros
# declaraciones = [
#     types.FunctionDeclaration(name=h.name, description=h.description, parameters_json_schema=h.input_schema)
#     for h in herramientas
# ]
#
# pregunta = "¿Hace frío ahora en Getafe? ¿Y a qué hora de hoy es más barata la luz?"
# respuesta = gemini.models.generate_content(
#     model="gemini-flash-latest",
#     contents=pregunta,
#     config=types.GenerateContentConfig(
#         tools=[types.Tool(function_declarations=declaraciones)],
#         automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
#     ),
# )
#
# # Lo que pide el modelo (no ejecuta nada: solo pide)
# for peticion in respuesta.function_calls or []:
#     print("El modelo pide:", peticion.name, dict(peticion.args))
#     print("El servidor contesta:", await llamar(peticion.name, dict(peticion.args)))

## Si vas por delante

### Extra A. Las otras dos cosas que ofrece un servidor MCP

Además de herramientas (que **hacen** algo), un servidor puede ofrecer **recursos** (algo que se **lee**) y **prompts** (plantillas que **elige la persona**, no la IA).

In [ ]:
async with conectar() as cliente:
    recursos = (await cliente.list_resources()).resources
    prompts = (await cliente.list_prompts()).prompts
    fuentes = await cliente.read_resource("madrid-ahora://fuentes")
    plan = await cliente.get_prompt("plan_para_hoy", {"lugar": "Getafe"})

print("RECURSOS:", [str(r.uri) for r in recursos])
print("PROMPTS:", [p.name for p in prompts])
print()
print(fuentes.contents[0].text)
print("EL PROMPT plan_para_hoy, RELLENO:")
print(plan.messages[0].content.text)

El prompt no ejecuta nada: es un texto que tu agente le pasaría al modelo. En Claude Code aparece como un comando que empieza por `/`.

### Extra B. ¿Dónde hace más frío ahora?

Cinco llamadas seguidas y una tabla ordenada. La ordena el código, no una IA: así no se equivoca.

In [ ]:
import pandas as pd

lugares = ["Madrid", "Getafe", "Alcobendas", "Navacerrada", "Aranjuez"]
filas = [await llamar("tiempo_ahora", {"lugar": lugar}) for lugar in lugares]
pd.DataFrame(filas)[["lugar", "temperatura_c", "sensacion_termica_c", "cielo", "viento_kmh"]].sort_values("temperatura_c")

Cambia la lista por los lugares que quieras.

### Extra C. Tu propio MCP

Cuando tengas tu MCP en Render, este cuaderno te sirve para probarlo sin agente:

1. Vuelve a la sección 2 y escribe **tu** URL (`https://...onrender.com/mcp`) y **tu** token.
2. Ejecuta la sección 3: ¿se entienden tus descripciones?
3. En la sección 4, llama a tus herramientas con `await llamar("tu_herramienta", {...})`.
4. En la sección 6, provoca un error en una de tus herramientas: ¿el mensaje dice qué hacer?

Si aquí funciona y en tu agente no, el fallo está en la configuración del agente, no en tu servidor.

## Si algo falla

| Pasa | Qué hacer |
|---|---|
| `ModuleNotFoundError: No module named 'mcp'` | Ejecuta la celda del `%pip install` de la sección 1 |
| `NameError: name 'URL' is not defined` | Ejecuta la celda de la sección 2 (URL y token) |
| La celda tarda mucho la primera vez | El servidor estaba dormido. Espera hasta un minuto |
| `401` | El token está mal copiado. Vuelve a escribirlo en la sección 2 |
| `ERROR. Esto es lo que leería la IA` | No es un fallo tuyo: es la herramienta avisando. Lee el mensaje |